# [A1] Human Annotation — CLIP Filter Validation

**Instructions:**
1. Set your `ANNOTATOR_NAME` below (for example, your initials).
2. Run all cells. The widget will show one figure at a time.
3. Click the category button that best describes the figure.
4. Use the **Notes** field for ambiguous cases (optional).
5. Your labels auto-save after each annotation. You can close and resume anytime.

**Categories:**
- **diagram**: schematic, flowchart, system architecture, pipeline, block diagram, neural network graph
- **plot**: statistical chart, line/scatter/bar plot, histogram, heatmap, confusion matrix, data visualization
- **photo**: natural photograph, camera/microscopy image, screenshot, pixel art, comic-style illustration
- **other**: dense text, standalone equations, tables of numbers, blank pages, anything else

In [ ]:
# ── CONFIG (edit this) ──────────────────────────────────────────
import os
ANNOTATOR_NAME = "annotator"  # names the output file annotations_<name>.jsonl
FIG_RAG_DIR = os.environ.get("FIG_RAG_DIR", os.path.expanduser("~/fig-rag-dir"))  # path to figure root

In [ ]:
import json
import os
from pathlib import Path
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

In [ ]:
# ── Load sample and existing annotations ──────────────────────
SCRIPT_DIR = Path(".").resolve()
SAMPLE_PATH = SCRIPT_DIR / "sample.jsonl"
OUTPUT_PATH = SCRIPT_DIR / f"annotations_{ANNOTATOR_NAME}.jsonl"

with open(SAMPLE_PATH) as f:
    samples = [json.loads(line) for line in f if line.strip()]

# Load any existing annotations (for resume)
annotations = {}
if OUTPUT_PATH.exists():
    with open(OUTPUT_PATH) as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                key = f"{rec['platform_id']}|{rec['figure_number']}"
                annotations[key] = rec

print(f"Annotator: {ANNOTATOR_NAME}")
print(f"Total figures: {len(samples)}")
print(f"Already annotated: {len(annotations)}")
print(f"Remaining: {len(samples) - len(annotations)}")

In [ ]:
# ── Helper: save one annotation ──────────────────────────────
def save_annotation(sample, label, notes=""):
    key = f"{sample['platform_id']}|{sample['figure_number']}"
    record = {
        **sample,
        f"label_{ANNOTATOR_NAME}": label,
        "notes": notes,
    }
    annotations[key] = record
    # Rewrite the full file (safe for small datasets)
    with open(OUTPUT_PATH, "w") as f:
        for rec in annotations.values():
            f.write(json.dumps(rec) + "\n")

In [ ]:
# ── Build the annotation widget ─────────────────────────────

# Find the first unannotated index
current_idx = [0]
for i, s in enumerate(samples):
    key = f"{s['platform_id']}|{s['figure_number']}"
    if key not in annotations:
        current_idx[0] = i
        break
else:
    current_idx[0] = len(samples)  # all done

# Output area
output = widgets.Output()

# Image display
image_widget = widgets.Image(format='png', width=600)

# Info label
info_label = widgets.HTML()

# Notes field
notes_field = widgets.Text(
    placeholder='Optional notes for ambiguous cases',
    layout=widgets.Layout(width='600px')
)

# Category buttons
CATEGORIES = ["diagram", "plot", "photo", "other"]
BUTTON_COLORS = {
    "diagram": "info",
    "plot": "success",
    "photo": "warning",
    "other": "danger",
}

buttons = []
for cat in CATEGORIES:
    btn = widgets.Button(
        description=cat,
        button_style=BUTTON_COLORS[cat],
        layout=widgets.Layout(width='140px', height='40px'),
    )
    buttons.append(btn)

# Navigation buttons
prev_btn = widgets.Button(description='< Prev', layout=widgets.Layout(width='80px'))
next_btn = widgets.Button(description='Next >', layout=widgets.Layout(width='80px'))
jump_field = widgets.IntText(value=current_idx[0] + 1, min=1, max=len(samples),
                             description='Go to #:', layout=widgets.Layout(width='180px'))

# Status bar
status_label = widgets.HTML()


def update_display():
    idx = current_idx[0]
    if idx >= len(samples):
        info_label.value = "<h3>All done! All 500 figures have been annotated.</h3>"
        image_widget.value = b''
        status_label.value = f"<b>Completed: {len(annotations)}/{len(samples)}</b>"
        return

    s = samples[idx]
    key = f"{s['platform_id']}|{s['figure_number']}"
    image_path = os.path.join(FIG_RAG_DIR, s["figure_path"])

    # Load image
    if os.path.exists(image_path):
        with open(image_path, 'rb') as f:
            image_widget.value = f.read()
    else:
        image_widget.value = b''

    # Show info — CLIP label and confidence are intentionally hidden to avoid bias
    existing_label = annotations.get(key, {}).get(f"label_{ANNOTATOR_NAME}", "—")
    existing_notes = annotations.get(key, {}).get("notes", "")
    notes_field.value = existing_notes

    info_label.value = (
        f"<b>Figure {idx+1} / {len(samples)}</b> &nbsp; | &nbsp; "
        f"Venue: {s['venue']} &nbsp; | &nbsp; "
        f"Paper: <code>{s['platform_id']}</code> Fig {s['figure_number']}"
        f"<br>Your label: <b>{existing_label}</b>"
    )

    jump_field.value = idx + 1
    done_count = len(annotations)
    status_label.value = f"<b>Annotated: {done_count}/{len(samples)}</b>"


def on_category_click(btn):
    idx = current_idx[0]
    if idx >= len(samples):
        return
    s = samples[idx]
    save_annotation(s, btn.description, notes_field.value)
    # Advance to next unannotated
    current_idx[0] = min(idx + 1, len(samples))
    notes_field.value = ""
    update_display()


def on_prev(btn):
    current_idx[0] = max(0, current_idx[0] - 1)
    update_display()


def on_next(btn):
    current_idx[0] = min(len(samples) - 1, current_idx[0] + 1)
    update_display()


def on_jump(change):
    current_idx[0] = max(0, min(len(samples) - 1, change['new'] - 1))
    update_display()


for btn in buttons:
    btn.on_click(on_category_click)
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
jump_field.observe(on_jump, names='value')

# Layout
button_row = widgets.HBox(buttons)
nav_row = widgets.HBox([prev_btn, next_btn, jump_field, status_label])

ui = widgets.VBox([
    info_label,
    image_widget,
    widgets.HTML("<b>Classify this figure:</b>"),
    button_row,
    notes_field,
    nav_row,
])

update_display()
display(ui)